# 2b. Feature Engineering

### Outline of this notebook:
* **Section 1:** Objective
* **Section 2:** Load Cleaned Dataset
* **Section 3:** Construct Engineered Features
  * 3.1: Ratio and interaction features
  * 3.2: Risk band features
  * 3.3: Macroeconomic variables
* **Section 4:** Save Feature-Enriched Dataset

---
## Section 1: Objective

This notebook constructs **engineered features** from the cleaned LendingClub dataset before modeling. The goal is to create variables that better capture economically meaningful dimensions of credit risk that raw variables do not directly express.

We construct three types of engineered features:

- 🔷 **Ratio and interaction features:** Combine raw variables to express relative financial burden, credit quality, and inquiry intensity
  - `income_to_loan_ratio` — loan size relative to income
  - `payment_to_income` — monthly repayment burden relative to monthly income
  - `fico_util_interaction` — interaction between FICO score and revolving utilization
  - `inq_per_credit_year` — inquiry intensity normalized by credit history length
  - `dti_x_int_rate` — interaction between DTI and interest rate (repayment pressure × cost)
  - `loan_to_installment` — implied term check (loan amount relative to monthly payment)
  - `delinq_rate` — delinquency frequency normalized by credit history length

- 🔷 **Risk band features:** Discretize continuous variables into interpretable categories for the linear baseline model
  - `fico_band` — FICO score risk tier (poor / fair / good / very good / excellent)
  - `dti_band` — debt-to-income ratio tier (very low / low / medium / high / very high)

- 🔷 **Macroeconomic variables:** Merge in monthly macro conditions at origination date
  - `unemployment_rate` — U.S. unemployment rate at loan origination (FRED: UNRATE)
  - `fed_funds_rate` — Federal funds rate at loan origination (FRED: FEDFUNDS)

The output is saved as `LendingClub_features.parquet` and used as the input for notebooks 3 and 4.

---
## Section 2: Load Cleaned Dataset

In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_parquet("../data/processed/LendingClub_cleaned.parquet")
print(df.shape)
df.head(3)

(1860765, 64)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,default,credit_age
0,5000.0,36.0,0.1065,162.87,B,B2,10.0,RENT,24000.0,Verified,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0,41.350685
1,2500.0,60.0,0.1527,59.83,C,C4,0.5,RENT,30000.0,Source Verified,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,1,27.095890
2,2400.0,36.0,0.1596,84.33,C,C5,10.0,RENT,12252.0,Not Verified,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0,24.506849


---
## Section 3: Construct Engineered Features

### 🔷 3.1: Ratio and interaction features

- **`income_to_loan_ratio`**: Annual income divided by loan amount. Higher values indicate the borrower is taking on a loan small relative to their income — generally lower risk.
- **`payment_to_income`**: Estimated monthly payment (`loan_amnt / term`) divided by monthly income. Captures the share of monthly income committed to loan repayment, a direct measure of repayment burden.
- **`fico_util_interaction`**: Product of FICO score (`fico_range_low`) and revolving utilization. Captures whether a borrower with a given credit score is also heavily utilizing credit — a higher utilization conditional on a given FICO may signal elevated risk.
- **`inq_per_credit_year`**: Number of credit inquiries in the last 6 months divided by credit history length in years. Normalizes inquiry activity by the length of credit history, capturing inquiry intensity rather than the raw count.
- **`dti_x_int_rate`**: Product of DTI and interest rate. Borrowers who are already highly leveraged (high DTI) and also face high borrowing costs are under compounded repayment pressure — this interaction captures that joint effect.
- **`loan_to_installment`**: Loan amount divided by monthly installment. This is essentially a function of term and interest rate; deviations from expected values may signal unusual loan structures or pricing.
- **`delinq_rate`**: Number of delinquencies in the past 2 years divided by credit history length. Normalizes past delinquency counts by exposure, so a borrower with 2 delinquencies in 5 years of credit history is treated differently from one with 2 delinquencies in 20 years.

In [8]:
out = df.copy()
eps = 1e-6

out["income_to_loan_ratio"] = out["annual_inc"] / (out["loan_amnt"] + eps)

out["payment_to_income"] = (
    out["loan_amnt"] / out["term"].clip(lower=1)
) / (out["annual_inc"] / 12 + eps)

out["fico_util_interaction"] = out["fico_range_low"] * out["revol_util"]

# credit_age is in years, so no /12 needed
out["inq_per_credit_year"] = out["inq_last_6mths"] / out["credit_age"].clip(lower=0.5)

out["dti_x_int_rate"] = out["dti"] * out["int_rate"]

out["loan_to_installment"] = out["loan_amnt"] / (out["installment"] + eps)

out["delinq_rate"] = out["delinq_2yrs"] / out["credit_age"].clip(lower=0.5)

In [9]:
# Quick sanity check
ratio_features = [
    "income_to_loan_ratio", "payment_to_income", "fico_util_interaction",
    "inq_per_credit_year", "dti_x_int_rate", "loan_to_installment", "delinq_rate"
]
out[ratio_features].describe()

,income_to_loan_ratio,payment_to_income,fico_util_interaction,inq_per_credit_year,dti_x_int_rate,loan_to_installment,delinq_rate
count,1.860764e+06,1.860764e+06,1.859366e+06,1.860763e+06,1.859655e+06,1.860764e+06,1.860764e+06
mean,7.412234e+00,2.439226e+05,3.482191e+02,2.467000e-02,2.529090e+00,3.291557e+01,1.181167e-02
std,1.392371e+01,1.196183e+07,1.675295e+02,3.781582e-02,2.322673e+00,5.829751e+00,3.393979e-02
min,0.000000e+00,5.376344e-05,0.000000e+00,0.000000e+00,-1.349000e-01,1.847882e+01,0.000000e+00
25%,3.428571e+00,3.642987e-02,2.226000e+02,0.000000e+00,1.256280e+00,2.947132e+01,0.000000e+00
50%,5.031447e+00,5.600000e-02,3.489200e+02,0.000000e+00,2.119208e+00,3.102218e+01,0.000000e+00
75%,8.200000e+00,8.130081e-02,4.761000e+02,4.193956e-02,3.341295e+00,3.320604e+01,0.000000e+00
max,6.200000e+03,1.111111e+09,6.201485e+03,4.544323e-01,3.095901e+02,2.322515e+03,1.471546e+00


### 🔷 3.2: Risk band features

- **`fico_band`**: FICO score bucketed into standard industry tiers. Uses `fico_range_low` as a proxy for the borrower's FICO score.
- **`dti_band`**: Debt-to-income ratio bucketed into risk tiers. Provides an interpretable ordinal version of `dti` for the linear baseline and descriptive analysis.

In [11]:
out["fico_band"] = pd.cut(
    out["fico_range_low"],
    bins=[300, 580, 670, 740, 800, 850],
    labels=["poor", "fair", "good", "very_good", "excellent"],
    include_lowest=True
)

out["dti_band"] = pd.cut(
    out["dti"],
    bins=[-np.inf, 10, 20, 35, 50, np.inf],
    labels=["very_low", "low", "medium", "high", "very_high"]
)

In [12]:
# Quick sanity check
print(out["fico_band"].value_counts().sort_index())
print()
print(out["dti_band"].value_counts().sort_index())

fico_band
poor               0
fair          469657
good         1201174
very_good     171819
excellent      18114
Name: count, dtype: int64

dti_band
very_low     339875
low          766883
medium       694592
high          50379
very_high      7926
Name: count, dtype: int64


### 🔷 3.3: Macroeconomic variables

Loan default is not purely a function of individual borrower characteristics — it is also affected by the **macroeconomic environment at the time of origination**. A borrower who takes out a loan during a recession faces structurally higher default risk than an otherwise identical borrower in a strong economy.

We merge two monthly macro series from FRED, matched to each loan's origination month (`issue_d`):
- **`unemployment_rate`** (FRED: `UNRATE`): The U.S. unemployment rate. Higher unemployment signals weaker labor market conditions and greater income risk for borrowers.
- **`fed_funds_rate`** (FRED: `FEDFUNDS`): The federal funds rate. Reflects the broader interest rate environment, which affects refinancing conditions and debt serviceability.

Both series are merged on the year-month of `issue_d`, so each loan receives the macro conditions prevailing at origination.

> **Note:** Requires `pandas_datareader`. Install with `pip install pandas-datareader` if not available.

In [14]:
import pandas_datareader.data as web

# Fetch monthly macro series from FRED
macro = web.DataReader(
    ['UNRATE', 'FEDFUNDS'],
    'fred',
    start='2007-01-01',
    end='2020-12-01'
).reset_index()

macro['year_month'] = macro['DATE'].dt.to_period('M')
macro = macro.drop(columns=['DATE'])

# Match each loan to its origination month
out['year_month'] = pd.to_datetime(out['issue_d']).dt.to_period('M')

out = out.merge(macro, on='year_month', how='left')
out = out.rename(columns={'UNRATE': 'unemployment_rate', 'FEDFUNDS': 'fed_funds_rate'})
out = out.drop(columns=['year_month'])

# Sanity check
print(out[['unemployment_rate', 'fed_funds_rate']].describe())
print(f"Missing unemployment_rate: {out['unemployment_rate'].isna().sum()}")
print(f"Missing fed_funds_rate:    {out['fed_funds_rate'].isna().sum()}")

       unemployment_rate  fed_funds_rate
count       1.860764e+06    1.860764e+06
mean        5.237689e+00    5.935803e-01
std         1.212262e+00    6.581111e-01
min         3.500000e+00    5.000000e-02
25%         4.400000e+00    1.200000e-01
50%         5.000000e+00    3.600000e-01
75%         5.700000e+00    9.100000e-01
max         1.480000e+01    5.260000e+00
Missing unemployment_rate: 1
Missing fed_funds_rate:    1


---
## Section 4: Save Feature-Enriched Dataset

In [16]:
all_new_cols = ratio_features + ["fico_band", "dti_band", "unemployment_rate", "fed_funds_rate"]

print("New features added:")
for col in all_new_cols:
    print(f"  {col}: {out[col].dtype}, {out[col].isna().sum()} missing")

New features added:
  income_to_loan_ratio: float64, 1 missing
  payment_to_income: float64, 1 missing
  fico_util_interaction: float64, 1399 missing
  inq_per_credit_year: float64, 2 missing
  dti_x_int_rate: float64, 1110 missing
  loan_to_installment: float64, 1 missing
  delinq_rate: float64, 1 missing
  fico_band: category, 1 missing
  dti_band: category, 1110 missing
  unemployment_rate: float64, 1 missing
  fed_funds_rate: float64, 1 missing


In [17]:
out.to_parquet("../data/processed/LendingClub_features.parquet", index=False)
print(f"Saved: {out.shape[0]:,} rows, {out.shape[1]} columns")

Saved: 1,860,765 rows, 75 columns
